# Notebook 3: From Polling to Change Data Capture (CDC)

The polling publisher from notebook 2 is simple and correct, but it has two costs:

- **Latency** - events wait for the next poll tick (often 100 ms - 1 s).
- **DB load** - repeated `SELECT` queries against the outbox table, even when there is nothing new.

**Change Data Capture (CDC)** takes a different approach: the database already writes every change to its Write-Ahead Log (WAL) for crash recovery. CDC *tails that log* and turns it into a stream of change events. No polling, near-real-time, and the application doesn't need to do anything special.

Postgres exposes WAL changes via **logical replication slots** - the exact mechanism **Debezium** uses in production.

## Setup

Logical decoding requires `wal_level=logical`. Our `docker-compose.yml` already sets that:

```yaml
command: ["postgres", "-c", "wal_level=logical"]
```

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```

## Step 1 - Create a logical replication slot

A slot is a named cursor into the WAL. Postgres will **retain WAL** until every slot has consumed up to the latest position. That's great (no lost events) and dangerous (an abandoned slot can fill your disk - we'll revisit this).

`test_decoding` is Postgres's built-in output plugin - human-readable, perfect for learning. Debezium uses `pgoutput` (binary, efficient) or `wal2json` (JSON).

In [1]:
import psycopg
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    # Idempotent: drop if it already exists from a previous run
    conn.execute("""
        SELECT pg_drop_replication_slot('demo_slot')
        WHERE EXISTS (SELECT 1 FROM pg_replication_slots WHERE slot_name='demo_slot')
    """)
    conn.execute("SELECT pg_create_logical_replication_slot('demo_slot', 'test_decoding')")
    slots = conn.execute('SELECT slot_name, plugin, active FROM pg_replication_slots').fetchall()
print('replication slots:', slots)

replication slots: [('demo_slot', 'test_decoding', False)]


## Step 2 - Produce some changes

Normal application traffic - an INSERT, an UPDATE, a DELETE. The slot is already capturing everything in the background.

In [2]:
with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS products')
    conn.execute('CREATE TABLE products (id SERIAL PRIMARY KEY, name TEXT, price INTEGER)')
    conn.execute("INSERT INTO products(name,price) VALUES ('book',25),('pen',5)")
    conn.execute("UPDATE products SET price=30 WHERE name='book'")
    conn.execute("DELETE FROM products WHERE name='pen'")
print('applied 2 inserts, 1 update, 1 delete')

applied 2 inserts, 1 update, 1 delete


## Step 3 - Drain the slot

`pg_logical_slot_get_changes` returns the decoded events and advances the slot. Each INSERT / UPDATE / DELETE becomes a structured row - **straight from the storage engine**, without the application writing to an outbox. That's CDC.

In [3]:
with psycopg.connect(DSN, autocommit=True) as conn:
    rows = conn.execute(
        "SELECT lsn, xid, data FROM pg_logical_slot_get_changes('demo_slot', NULL, NULL)"
    ).fetchall()
for lsn, xid, data in rows:
    print(f'{lsn}  xid={xid}  {data}')

0/1C08230  xid=945  BEGIN 945
0/1C0AB68  xid=945  COMMIT 945
0/1C0AB68  xid=946  BEGIN 946
0/1C110E8  xid=946  COMMIT 946
0/1C110E8  xid=947  BEGIN 947
0/1C11150  xid=947  table public.products: INSERT: id[integer]:1 name[text]:'book' price[integer]:25
0/1C11238  xid=947  table public.products: INSERT: id[integer]:2 name[text]:'pen' price[integer]:5
0/1C112F0  xid=947  COMMIT 947
0/1C112F0  xid=948  BEGIN 948
0/1C112F0  xid=948  table public.products: UPDATE: id[integer]:1 name[text]:'book' price[integer]:30
0/1C11378  xid=948  COMMIT 948
0/1C11378  xid=949  BEGIN 949
0/1C11378  xid=949  table public.products: DELETE: id[integer]:2
0/1C113E8  xid=949  COMMIT 949


## Peek vs get

`pg_logical_slot_get_changes` *consumes* the events (advances the slot). `pg_logical_slot_peek_changes` returns them **without** advancing - useful when the downstream sink hasn't confirmed delivery yet. In real consumers (Debezium, custom workers) you read with peek-like semantics and only advance after the sink acks. That's how CDC achieves **at-least-once** end-to-end.

In [4]:
# A second drain returns nothing - the slot already advanced
with psycopg.connect(DSN, autocommit=True) as conn:
    rows = conn.execute(
        "SELECT lsn, data FROM pg_logical_slot_get_changes('demo_slot', NULL, NULL)"
    ).fetchall()
print('second drain returned', len(rows), 'rows')

second drain returned 0 rows


## The #1 production footgun: abandoned slots

Postgres will **keep every WAL segment** a slot still needs. If your consumer is down for hours (or you forgot to drop a test slot), the WAL directory grows until it fills the disk and the database stops accepting writes.

Always monitor:

- `pg_replication_slots.active` - is the consumer connected?
- `pg_current_wal_lsn() - confirmed_flush_lsn` - how many bytes behind is the slot?

And always clean up slots you don't need any more:

In [5]:
with psycopg.connect(DSN, autocommit=True) as conn:
    lag = conn.execute('''
        SELECT slot_name,
               active,
               pg_size_pretty(pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)) AS lag
        FROM pg_replication_slots
    ''').fetchall()
    print('slot lag:', lag)
    # Cleanup for this demo so re-running is safe and nothing keeps WAL pinned
    conn.execute("SELECT pg_drop_replication_slot('demo_slot')")
print('slot dropped - no WAL retention leak')

slot lag: [('demo_slot', False, '0 bytes')]
slot dropped - no WAL retention leak


## Polling outbox vs logical CDC

| | Polling outbox | Logical CDC |
|---|---|---|
| Application code | small `INSERT` into outbox | none - DB does it |
| Latency | poll interval (100 ms - 1 s) | milliseconds |
| DB load | repeated reads of outbox | continuous WAL streaming |
| Event shape | whatever you put in the payload | raw row changes (INSERT/UPDATE/DELETE) |
| Ops complexity | one cron / worker | replication slot, lag monitoring, Debezium/Kafka Connect |
| Portability | any SQL database | DB-specific (WAL format, plugins) |

**The hybrid pattern (very common in production):** keep the `outbox` table from notebook 2, but ship it with CDC instead of polling. You get the clean domain event shape of the outbox *and* the low latency of WAL streaming. Debezium even has a built-in [Outbox Event Router](https://debezium.io/documentation/reference/transformations/outbox-event-router.html) for exactly this.

Rule of thumb:

- Start with **polling outbox** - it's ~50 lines of code and enough for most teams.
- Graduate to **CDC + outbox** when you need sub-second fan-out or many consumers.
- Use **raw-table CDC (no outbox)** only when downstream actually wants row-level changes (analytics, search indexes, caches).